In [ ]:
import re
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.stattools import adfuller, kpss

from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import VECM, select_coint_rank
from statsmodels.tsa.ardl import ARDL, UECM
from statsmodels.tsa.statespace.structural import UnobservedComponents

import warnings
warnings.filterwarnings('ignore')

In [ ]:

path = 'done_diplom_filled_v2_with_keyrate.xlsx'
df_raw = pd.read_excel(path)
print(df_raw.shape)
df_raw.head()

#просто чтобы потом сто раз не писать долгие
explicit_map = {'Дата': 'date',
    'Задолженность по предоставленным кредитам, млн руб., в том числе': 'total_debt',
    'просроченная задолженность по предоставленным кредитам, млн руб.': 'overdue_debt',
    'Средневзвешенная ставка по кредитам, выданным в течение месяца, %': 'avg_loan_rate',
    'Ключевая ставка, % годовых': 'cb_rate'}
used_names = set(explicit_map.values())

def sanitize_name(name: str) -> str:
    name = str(name).strip().lower()
    name = name.replace('%', 'pct')
    name = name.replace('№', 'num')
    name = re.sub(r'[^0-9a-zA-Zа-яА-Я_]+', '_', name)
    name = re.sub(r'_+', '_', name).strip('_')
    if not name:
        name = 'col'
    return name

rename_map = {}
for col in df_raw.columns:
    if col in explicit_map:
        rename_map[col] = explicit_map[col]
    else:
        base = sanitize_name(col)
        candidate = base
        counter = 1
        while candidate in used_names:
            counter += 1
            candidate = f"{base}_{counter}"
        rename_map[col] = candidate
        used_names.add(candidate)

df = df_raw.rename(columns=rename_map).copy()

df['date'] = pd.to_datetime(df['date'])
# на всякий слукчай чтобды проблем не было
df = df.sort_values('date').reset_index(drop=True)

for col in df.columns:
    if col == 'date':
        continue
    df[col] = pd.to_numeric(
        df[col].astype(str)
              .str.replace(',', '.', regex=False)
              .str.replace('%', '', regex=False)
              .str.replace('\xa0', '', regex=False)
              .str.strip(),
        errors='coerce'
    )
# собственно сам таргет который я обозначил в работе
df['target_ratio'] = df['overdue_debt'] / df['total_debt']
df['rate_spread'] = df['avg_loan_rate'] - df['cb_rate']

# тоже обозночал в работе почему именно ffil бралд
numeric_cols = [c for c in df.columns if c != 'date']
df[numeric_cols] = df[numeric_cols].ffill()

df = df.dropna(subset=['target_ratio', 'rate_spread']).copy()

series = df.set_index('date')

In [ ]:
selection_n_train = 85
selection_sample = series.iloc[:selection_n_train].copy()

excluded_from_selection = {'target_ratio','rate_spread'}

candidate_features = [col for col in selection_sample.columns if col not in excluded_from_selection and pd.api.types.is_numeric_dtype(selection_sample[col])]

X_full = selection_sample[candidate_features].copy()
y_full = selection_sample['target_ratio'].copy()
X_full = X_full.replace([np.inf, -np.inf], np.nan)
y_full = y_full.replace([np.inf, -np.inf], np.nan)

valid_cols = []
for col in X_full.columns:
    s = X_full[col]
    if s.notna().sum() == 0:
        continue
    if s.nunique(dropna=True) <= 1:
        continue
    valid_cols.append(col)
X_full = X_full[valid_cols].copy()
valid_rows = X_full.notna().all(axis=1) & y_full.notna()
X_full = X_full.loc[valid_rows].copy()
y_full = y_full.loc[valid_rows].copy()


base_splitter = TimeSeriesSplit(n_splits=5)
base_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],cv=base_splitter,random_state=0,max_iter=50000))])
base_pipeline.fit(X_full, y_full)
base_model = base_pipeline.named_steps['enet']
start_fracs = [0.60, 0.70, 0.80, 0.90, 1.00]
selection_records = []
for frac in start_fracs:
    end_idx = max(30, int(len(X_full) * frac))
    X_sub = X_full.iloc[:end_idx].copy()
    y_sub = y_full.iloc[:end_idx].copy()
    if len(X_sub) < 25:
        continue

    n_splits_sub = min(5, max(2, len(X_sub) // 12))
    sub_splitter = TimeSeriesSplit(n_splits=n_splits_sub)
    sub_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('enet', ElasticNetCV(l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],cv=base_splitter,random_state=0,max_iter=50000))])
    sub_pipeline.fit(X_sub, y_sub)
    sub_model = sub_pipeline.named_steps['enet']
    coef = pd.Series(sub_model.coef_, index=X_sub.columns)
    selection_records.append(pd.DataFrame({'feature': coef.index,'coef': coef.values,'abs_coef': coef.abs().values,'selected': (coef.abs() > 1e-10).astype(int),'window_frac': frac}))

stability_df = pd.concat(selection_records, ignore_index=True)
stability_summary = (stability_df.groupby('feature', as_index=False).agg(selection_frequency=('selected', 'mean'),mean_abs_coef=('abs_coef', 'mean'),max_abs_coef=('abs_coef', 'max')))

base_coef_df=pd.DataFrame({'feature': X_full.columns,'base_coef': base_model.coef_,'base_abs_coef': np.abs(base_model.coef_)})

feature_ranking = (stability_summary.merge(base_coef_df, on='feature', how='left').sort_values(['selection_frequency', 'mean_abs_coef', 'base_abs_coef'],ascending=[False, False, False]).reset_index(drop=True))

selection_eval_candidates = sorted(set([
    min(4, len(feature_ranking)),
    min(6, len(feature_ranking)),
    min(8, len(feature_ranking)),
    min(10, len(feature_ranking))
]))
selection_eval_candidates = [k for k in selection_eval_candidates if k >= 3]

selection_eval_df = selection_sample[['target_ratio', 'rate_spread'] + feature_ranking['feature'].tolist()].copy()
selection_eval_df = selection_eval_df.replace([np.inf, -np.inf], np.nan).ffill().dropna()

selection_val_size = min(max(8, len(selection_eval_df) // 6), 15)
selection_fit_df = selection_eval_df.iloc[:-selection_val_size].copy()
selection_val_df = selection_eval_df.iloc[-selection_val_size:].copy()

selection_size_records = []
selection_size_candidates = []

for top_k in selection_eval_candidates:
    feat_set = ['rate_spread'] + feature_ranking.head(top_k)['feature'].tolist()

    for d_order in [0, 1]:
        try:
            model = SARIMAX(selection_fit_df['target_ratio'],exog=selection_fit_df[feat_set],order=(1, d_order, 0),seasonal_order=(0, 0, 0, 0),trend='c',enforce_stationarity=False,enforce_invertibility=False)
            res = model.fit(disp=False, maxiter=200)
            pred = pd.Series(
                res.forecast(steps=len(selection_val_df), exog=selection_val_df[feat_set]),
                index=selection_val_df.index)
            score = np.sqrt(mean_squared_error(selection_val_df['target_ratio'],pred))

            selection_size_records.append({'top_k_features': top_k,'d_order': d_order,'rmse': score,'aic': getattr(res, 'aic', np.nan)})
            selection_size_candidates.append({'top_k_features': top_k,'d_order': d_order,'rmse': score})
        except Exception:
            continue

if not selection_size_candidates:
    best_feature_count = min(8, len(feature_ranking))
else:
    selection_size_tuning_df = (pd.DataFrame(selection_size_records).sort_values(['rmse', 'aic']).reset_index(drop=True))
    best_feature_count = int(selection_size_tuning_df.iloc[0]['top_k_features'])

selected_core_features = feature_ranking.head(best_feature_count)['feature'].tolist()
selected_features = ['rate_spread'] + selected_core_features




display(feature_ranking.head(15))



In [ ]:
model_cols = ['target_ratio'] + selected_features
model_df = series[model_cols].copy()
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.ffill()
model_df = model_df.dropna(axis=0, how='any').copy()
display(model_df.head())

In [ ]:
n_train = 85
train = model_df.iloc[:n_train].copy()
test  = model_df.iloc[n_train:].copy()
endog_train = train['target_ratio'].copy()
endog_test  = test['target_ratio'].copy()
exog_train = train[selected_features].copy()
exog_test  = test[selected_features].copy()

# тут в модеи все эндогены так чтот так
var_cols = ['target_ratio'] + selected_features
var_train = train[var_cols].copy()
var_test  = test[var_cols].copy()
print( exog_train.shape)
print( exog_test.shape)

In [ ]:
VALIDATION_SIZE = min(max(10, len(train) // 6), 18)
MAX_TUNE_LAG = min(6, max(2, len(train) // 10))

train_fit = train.iloc[:-VALIDATION_SIZE].copy()
train_val = train.iloc[-VALIDATION_SIZE:].copy()
print(f"тренировочная выборка {len(train_fit)}")
print(f"тренировочная выборка {len(train_val)}")




def safe_kpss(series):
        stat, pvalue, _, _ = kpss(series, regression='c', nlags='auto')
        return stat, pvalue


def stationarity_test(series):
    s = pd.Series(series).dropna().astype(float)
    if len(s) < 12:
        return {'adf_pvalue': np.nan,'kpss_pvalue': np.nan,'stationary': False}

    try:
        adf_pvalue = adfuller(s, autolag='AIC')[1]
    except Exception:
        adf_pvalue = np.nan

    _, kpss_pvalue = safe_kpss(s)

    stationary = False
    if not np.isnan(adf_pvalue) and not np.isnan(kpss_pvalue):
        stationary = (adf_pvalue < 0.05) and (kpss_pvalue > 0.05)
    elif not np.isnan(adf_pvalue):
        stationary = adf_pvalue < 0.05

    return {'adf_pvalue': adf_pvalue,'kpss_pvalue': kpss_pvalue,'stationary': stationary}


def choose_integration_order(series, max_d=2):
    current = pd.Series(series).dropna().astype(float)
    diagnostics = []

    for d in range(max_d + 1):
        stat = stationarity_test(current)
        diagnostics.append({'diff_order': d,'adf_pvalue': stat['adf_pvalue'],'kpss_pvalue': stat['kpss_pvalue'],'stationary': stat['stationary']})
        if stat['stationary']:
            return d, pd.DataFrame(diagnostics)

        current = current.diff().dropna()

    return max_d, pd.DataFrame(diagnostics)


def make_stationarity_table(df, max_d=1):
    rows = []
    for col in df.columns:
        level_stat = stationarity_test(df[col])
        chosen_d, diag_df = choose_integration_order(df[col], max_d=max_d)
        rows.append({'series': col,'level_adf_pvalue': level_stat['adf_pvalue'],'level_kpss_pvalue': level_stat['kpss_pvalue'],'stationary_in_levels': level_stat['stationary'],'chosen_diff_order': chosen_d})
    return pd.DataFrame(rows).sort_values(['chosen_diff_order', 'series']).reset_index(drop=True)



def build_nan_forecast(index, fill_value=np.nan):
    return pd.Series(fill_value, index=index, dtype=float)


def difference_series_train_test(train_series, test_series, diff_order=0):
    train_series = pd.Series(train_series).astype(float).copy()
    test_series = pd.Series(test_series).astype(float).copy()

    if diff_order <= 0:
        return train_series, test_series

    combined = pd.concat([train_series, test_series], axis=0)
    transformed = combined.copy()
    for _ in range(int(diff_order)):
        transformed = transformed.diff()

    train_out = transformed.loc[train_series.index].dropna().astype(float)
    test_out = transformed.loc[test_series.index].dropna().astype(float)
    return train_out, test_out


def build_stationary_train_test(train_df, test_df, diff_order_map):
    train_parts = []
    test_parts = []

    for col in train_df.columns:
        d = int(diff_order_map.get(col, 0))
        tr_col, te_col = difference_series_train_test(train_df[col], test_df[col], diff_order=d)
        train_parts.append(tr_col.rename(col))
        test_parts.append(te_col.rename(col))

    train_out = pd.concat(train_parts, axis=1).dropna()
    test_out = pd.concat(test_parts, axis=1).dropna()
    return train_out, test_out


def align_target_and_exog(target_series, exog_df):
    idx = pd.Index(target_series.index).intersection(exog_df.index)
    return target_series.loc[idx].astype(float), exog_df.loc[idx].astype(float)


def safe_metric_frame(y_true, y_pred):
    tmp = pd.concat([pd.Series(y_true, name='y_true').astype(float),pd.Series(y_pred, name='y_pred').astype(float)], axis=1).dropna()
    return tmp


def forecast_from_differences(last_level, diff_forecast):
    levels = []
    current = float(last_level)
    for value in np.asarray(diff_forecast, dtype=float).ravel():
        current += value
        levels.append(current)
    return np.array(levels, dtype=float)


def fit_sarimax_candidate(y_train, X_train, y_valid, X_valid, order, seasonal_order=(0, 0, 0, 0), trend='c'):
        model = SARIMAX(y_train,exog=X_train,order=order, seasonal_order=seasonal_order,trend=trend,enforce_stationarity=False,enforce_invertibility=False)
        res = model.fit(disp=False, maxiter=200)
        pred = pd.Series(res.forecast(steps=len(y_valid), exog=X_valid),index=y_valid.index).astype(float)

        return {'order': order,'seasonal_order': seasonal_order,'trend': trend,'rmse': rmse(y_valid, pred),'aic': getattr(res, 'aic', np.nan),'bic': getattr(res, 'bic', np.nan),'result': res,'forecast': pred}

def tune_arimax_sarimax(endog_train, exog_train):
    y_fit = endog_train.iloc[:-VALIDATION_SIZE]
    y_val = endog_train.iloc[-VALIDATION_SIZE:]
    X_fit = exog_train.iloc[:-VALIDATION_SIZE]
    X_val = exog_train.iloc[-VALIDATION_SIZE:]

    d_target, d_diag = choose_integration_order(y_fit, max_d=1)
    d_candidates = sorted(set([0, 1, min(1, d_target)]))

    arima_orders = [(p, d, q) for p, d, q in product(range(0, 4), d_candidates, range(0, 4)) if (p + q) <= 4 and not (p == 0 and d == 0 and q == 0)]

    arimax_records = []
    arimax_candidates = []

    for order in arima_orders:
        result = fit_sarimax_candidate(y_fit, X_fit, y_val, X_val,order=order,seasonal_order=(0, 0, 0, 0),trend='c')
        arimax_records.append({'order': result['order'],'seasonal_order': result['seasonal_order'],'rmse': result['rmse'],'aic': result['aic'],'bic': result['bic'],
            'status': 'ok' if np.isfinite(result['rmse']) else 'failed'})
        if np.isfinite(result['rmse']):
            arimax_candidates.append(result)



    arimax_tuning_df = pd.DataFrame(arimax_records).sort_values(['rmse', 'aic']).reset_index(drop=True)
    best_arimax_spec = min(arimax_candidates, key=lambda item: (item['rmse'], item['aic']))

    top_orders = []
    for order in arimax_tuning_df.loc[arimax_tuning_df['status'] == 'ok', 'order'].tolist():
        if order not in top_orders:
            top_orders.append(order)
        if len(top_orders) >= 4:
            break

    seasonal_candidates = [(0, 0, 0, 0),(1, 0, 0, 12),(0, 1, 1, 12),(1, 1, 0, 12),(1, 0, 0, 24),(0, 1, 1, 24)]

    sarimax_records = []
    sarimax_candidates = []

    for order in top_orders:
        for seasonal_order in seasonal_candidates:
            result = fit_sarimax_candidate(
                y_fit, X_fit, y_val, X_val,
                order=order,
                seasonal_order=seasonal_order,
                trend='c'
            )
            sarimax_records.append({'order': result['order'],'seasonal_order': result['seasonal_order'],'rmse': result['rmse'],'aic': result['aic'],'bic': result['bic'],
                'status': 'ok' if np.isfinite(result['rmse']) else 'failed'})
            if np.isfinite(result['rmse']):
                sarimax_candidates.append(result)

    if sarimax_candidates:
        sarimax_tuning_df = pd.DataFrame(sarimax_records).sort_values(['rmse', 'aic']).reset_index(drop=True)
        best_sarimax_spec = min(sarimax_candidates, key=lambda item: (item['rmse'], item['aic']))
    else:
        sarimax_tuning_df = pd.DataFrame(sarimax_records)
        best_sarimax_spec = {'order': best_arimax_spec['order'],'seasonal_order': (0, 0, 0, 0),'trend': 'c'}

    return best_arimax_spec, best_sarimax_spec, arimax_tuning_df, sarimax_tuning_df, d_diag



def get_var_ic_lag_candidates(train_block, system_cols, diff_order=0, maxlags=None):
    if maxlags is None:
        maxlags = MAX_TUNE_LAG
    try:
        if diff_order == 0:
            data = train_block[system_cols].copy().dropna()
        else:
            data = train_block[system_cols].diff(diff_order).dropna()
        if len(data) <= 8:
            return [1]

        sel = VAR(data).select_order(maxlags=min(maxlags, max(1, len(data)//4)))
        raw_vals = []
        for attr in ['aic', 'bic', 'hqic', 'fpe']:
            val = getattr(sel, attr, None)
            if val is not None and pd.notna(val):
                try:
                    raw_vals.append(int(val))
                except Exception:
                    pass

        cands = set([1])
        for v in raw_vals:
            cands.update([max(1, v-1), v, v+1])

        cands = sorted([lag for lag in cands if 1 <= lag <= min(maxlags, max(1, len(data)//4))])
        return cands or [1]
    except Exception:
        return list(range(1, maxlags + 1))


def fit_var_candidate(train_block, valid_block, system_cols, lag_order, diff_order=0):
    try:
        if diff_order == 0:
            model_data = train_block[system_cols].copy().dropna()
            if len(model_data) <= lag_order + 2:
                return None
            res = VAR(model_data).fit(lag_order)
            pred = res.forecast(model_data.values[-res.k_ar:], steps=len(valid_block))
            target_forecast = pd.Series(pred[:, 0], index=valid_block.index).astype(float)
        else:
            model_data = train_block[system_cols].diff(diff_order).dropna()
            if len(model_data) <= lag_order + 2:
                return None
            res = VAR(model_data).fit(lag_order)
            pred_diff = res.forecast(model_data.values[-res.k_ar:], steps=len(valid_block))
            target_forecast = pd.Series(forecast_from_differences(train_block['target_ratio'].iloc[-1], pred_diff[:, 0]),index=valid_block.index).astype(float)

        aligned = safe_metric_frame(valid_block['target_ratio'], target_forecast)
        if aligned.empty:
            return None

        return {'lag_order': lag_order,'diff_order': diff_order,'rmse': float(np.sqrt(mean_squared_error(aligned['y_true'], aligned['y_pred']))),'aic': getattr(res, 'aic', np.nan),'bic': getattr(res, 'bic', np.nan),
            'hqic': getattr(res, 'hqic', np.nan),'is_stable': bool(res.is_stable()),'result': res,'forecast': target_forecast}
    except Exception:
        return None


def tune_var_system(var_train_df, selected_exog):
    system_cols = ['target_ratio'] + selected_exog
    y_fit = var_train_df.iloc[:-VALIDATION_SIZE].copy()
    y_val = var_train_df.iloc[-VALIDATION_SIZE:].copy()

    stationarity_df = make_stationarity_table(y_fit[system_cols], max_d=1)
    nonstationary_share = 1 - stationarity_df['stationary_in_levels'].mean()

    diff_candidates = [0, 1]
    records = []
    candidates = []

    for diff_order in diff_candidates:
        lag_candidates = get_var_ic_lag_candidates(y_fit, system_cols, diff_order=diff_order, maxlags=MAX_TUNE_LAG)
        for lag_order in lag_candidates:
            fitted = fit_var_candidate(y_fit, y_val, system_cols, lag_order=lag_order, diff_order=diff_order)
            if fitted is None:
                continue

            records.append({'lag_order': lag_order,'diff_order': diff_order,'rmse': fitted['rmse'],'aic': fitted['aic'],'bic': fitted['bic'],'hqic': fitted['hqic'],'is_stable': fitted['is_stable']})
            candidates.append(fitted)


    tuning_df = pd.DataFrame(records).sort_values(['rmse', 'bic', 'hqic']).reset_index(drop=True)

    stable_candidates = [c for c in candidates if c['is_stable']]
    if stable_candidates:
        best_candidate = min(stable_candidates, key=lambda item: (item['rmse'], item['bic'], item['hqic']))
    else:
        best_candidate = min(candidates, key=lambda item: (item['rmse'], item['bic'], item['hqic']))

    diagnostics = {'stationarity_table': stationarity_df,'nonstationary_share': nonstationary_share,'level_var_stable_available': any((row['diff_order'] == 0) and bool(row['is_stable']) for row in records)}

    return best_candidate, tuning_df, diagnostics



def get_vecm_rank_candidates(train_block, system_cols, k_ar_diff, max_rank):
    rank_candidates = set([1])
    try:
        rank_obj = select_coint_rank(train_block[system_cols], det_order=0, k_ar_diff=k_ar_diff, method='trace', signif=0.05)
        suggested_rank = int(rank_obj.rank)
        for r in [suggested_rank - 1, suggested_rank, suggested_rank + 1]:
            if 1 <= r <= max_rank:
                rank_candidates.add(r)
    except Exception:
        pass
    return sorted(rank_candidates)


def fit_vecm_candidate(train_block, valid_block, system_cols, k_ar_diff, coint_rank, deterministic='co'):
    try:
        model = VECM(train_block[system_cols], k_ar_diff=k_ar_diff, coint_rank=coint_rank, deterministic=deterministic)
        res = model.fit()
        pred = res.predict(steps=len(valid_block))
        target_forecast = pd.Series(pred[:, 0], index=valid_block.index).astype(float)
        aligned = safe_metric_frame(valid_block['target_ratio'], target_forecast)
        if aligned.empty:
            return None
        return {'k_ar_diff': k_ar_diff,'coint_rank': coint_rank,'rmse': float(np.sqrt(mean_squared_error(aligned['y_true'], aligned['y_pred']))),'result': res,'forecast': target_forecast}
    except Exception:
        return None


def tune_vecm_system(var_train_df, selected_exog):
    system_cols = ['target_ratio'] + selected_exog
    y_fit = var_train_df.iloc[:-VALIDATION_SIZE].copy()
    y_val = var_train_df.iloc[-VALIDATION_SIZE:].copy()

    max_rank = min(3, len(system_cols) - 1)

    try:
        level_stationarity = make_stationarity_table(y_fit[system_cols], max_d=1)
        all_stationary = bool(level_stationarity['stationary_in_levels'].all())
    except Exception:
        pass

    k_candidates = sorted(set([1, 2, min(3, MAX_TUNE_LAG)]))
    records = []
    candidates = []

    for k_ar_diff in k_candidates:
        rank_candidates = get_vecm_rank_candidates(y_fit, system_cols, k_ar_diff=k_ar_diff, max_rank=max_rank)
        for coint_rank in rank_candidates:
            fitted = fit_vecm_candidate(y_fit, y_val, system_cols,k_ar_diff=k_ar_diff,coint_rank=coint_rank,deterministic='co')
            if fitted is None:
                continue

            records.append({'k_ar_diff': k_ar_diff,'coint_rank': coint_rank,'rmse': fitted['rmse']})
            candidates.append(fitted)

    tuning_df = pd.DataFrame(records).sort_values(['rmse', 'k_ar_diff', 'coint_rank']).reset_index(drop=True)
    best_candidate = min(candidates, key=lambda item: item['rmse'])
    return best_candidate, tuning_df




def fit_ardl_candidate(y_train, X_train, y_valid, X_valid, p_lag, q_lag, causal, feat_cols=None):
    try:
        if feat_cols is None:
            feat_cols = list(X_train.columns)
        X_train_sub = X_train[feat_cols].copy()
        X_valid_sub = X_valid[feat_cols].copy()
        order_dict = {feature: q_lag for feature in X_train_sub.columns}
        model = ARDL(y_train,lags=p_lag,exog=X_train_sub,order=order_dict,trend='c',causal=causal)
        res = model.fit()
        pred = pd.Series(
            res.forecast(steps=len(y_valid), exog=X_valid_sub),
            index=y_valid.index
        )
        return {'p_lag': p_lag,'q_lag': q_lag,'causal': causal,'feat_cols': feat_cols,'rmse': rmse(y_valid, pred),'aic': getattr(res, 'aic', np.nan),'bic': getattr(res, 'bic', np.nan),'model': model,
            'result': res,'forecast': pred}
    except Exception:
        return None


def tune_ardl(endog_train, exog_train):
    y_fit = endog_train.iloc[:-VALIDATION_SIZE]
    y_val = endog_train.iloc[-VALIDATION_SIZE:]
    X_fit = exog_train.iloc[:-VALIDATION_SIZE]
    X_val = exog_train.iloc[-VALIDATION_SIZE:]

    p_candidates = list(range(1, min(3, MAX_TUNE_LAG) + 1))
    q_candidates = [0, 1, 2]
    causal_candidates = [True, False]
    feat_count_candidates = sorted(set([min(2, X_fit.shape[1]), min(3, X_fit.shape[1]), min(4, X_fit.shape[1]), X_fit.shape[1]]))
    feat_count_candidates = [k for k in feat_count_candidates if k >= 1]

    records = []
    candidates = []

    for top_k, p_lag, q_lag, causal in product(feat_count_candidates, p_candidates, q_candidates, causal_candidates):
        feat_cols = list(X_fit.columns[:top_k])
        fitted = fit_ardl_candidate(y_fit, X_fit, y_val, X_val, p_lag=p_lag, q_lag=q_lag, causal=causal, feat_cols=feat_cols)
        if fitted is None:
            continue

        records.append({
            'n_features': top_k,'features': ', '.join(feat_cols),'p_lag': p_lag,'q_lag': q_lag,'causal': causal,'uecm_compatible': q_lag >= 1,'rmse': fitted['rmse'],'aic': fitted['aic'],'bic': fitted['bic']})
        candidates.append(fitted)


    tuning_df = pd.DataFrame(records).sort_values(['rmse', 'bic']).reset_index(drop=True)
    best_candidate = min(candidates, key=lambda item: (item['rmse'], item['bic']))
    return best_candidate, tuning_df


def fit_uecm_candidate(y_train, X_train, y_valid, X_valid, p_lag, q_lag, causal, feat_cols=None):
    if q_lag < 1:
        return None

    try:
        if feat_cols is None:
            feat_cols = list(X_train.columns)
        X_train_sub = X_train[feat_cols].copy()
        X_valid_sub = X_valid[feat_cols].copy()
        order_dict = {feature: q_lag for feature in X_train_sub.columns}
        ardl_model = ARDL( y_train,lags=p_lag,exog=X_train_sub,order=order_dict,trend='c',causal=causal)
        uecm_model = UECM.from_ardl(ardl_model)
        uecm_res = uecm_model.fit()
        pred = pd.Series(uecm_res.forecast(steps=len(y_valid), exog=X_valid_sub),index=y_valid.index)
        return {'p_lag': p_lag,'q_lag': q_lag,'causal': causal,'feat_cols': feat_cols,'rmse': rmse(y_valid, pred),'aic': getattr(uecm_res, 'aic', np.nan),'bic': getattr(uecm_res, 'bic', np.nan),'model': uecm_model,
            'result': uecm_res,'forecast': pred}
    except Exception:
        return None


def tune_uecm(y_train, X_train, y_valid, X_valid, p_lag, q_lag, causal):
    if q_lag < 1:
        return None

    try:
        order_dict = {feature: q_lag for feature in X_train.columns}
        ardl_model = ARDL(y_train,lags=p_lag,exog=X_train,order=order_dict,trend='c',causal=causal)
        uecm_model = UECM.from_ardl(ardl_model)
        uecm_res = uecm_model.fit()
        pred = pd.Series(uecm_res.forecast(steps=len(y_valid), exog=X_valid),index=y_valid.index)
        return {'p_lag': p_lag,'q_lag': q_lag,'causal': causal,'rmse': rmse(y_valid, pred),'aic': getattr(uecm_res, 'aic', np.nan),'bic': getattr(uecm_res, 'bic', np.nan),'model': uecm_model,
            'result': uecm_res,'forecast': pred}
    except Exception:
        return None



def tune_uecm(endog_train, exog_train):
    y_fit = endog_train.iloc[:-VALIDATION_SIZE]
    y_val = endog_train.iloc[-VALIDATION_SIZE:]
    X_fit = exog_train.iloc[:-VALIDATION_SIZE]
    X_val = exog_train.iloc[-VALIDATION_SIZE:]

    p_candidates = list(range(1, min(3, MAX_TUNE_LAG) + 1))
    q_candidates = [1, 2]
    causal_candidates = [True, False]
    feat_count_candidates = sorted(set([min(1, X_fit.shape[1]), min(2, X_fit.shape[1]), min(3, X_fit.shape[1])]))
    feat_count_candidates = [k for k in feat_count_candidates if k >= 1]

    records = []
    candidates = []

    for top_k, p_lag, q_lag, causal in product(feat_count_candidates, p_candidates, q_candidates, causal_candidates):
        feat_cols = list(X_fit.columns[:top_k])
        fitted = fit_uecm_candidate(y_fit, X_fit, y_val, X_val, p_lag=p_lag, q_lag=q_lag, causal=causal, feat_cols=feat_cols)
        if fitted is None:
            continue

        records.append({'n_features': top_k,'features': ', '.join(feat_cols),'p_lag': p_lag,'q_lag': q_lag,'causal': causal,'rmse': fitted['rmse'],'aic': fitted['aic'],'bic': fitted['bic']})
        candidates.append(fitted)



    tuning_df = pd.DataFrame(records).sort_values(['rmse', 'bic']).reset_index(drop=True)
    best_candidate = min(candidates, key=lambda item: (item['rmse'], item['bic']))
    return best_candidate, tuning_df



def fit_ucm_candidate(y_train, X_train, y_valid, X_valid, trend_name, seasonal_period):
    try:
        kwargs = {'endog': pd.Series(y_train).astype(float),'exog': pd.DataFrame(X_train).astype(float),'level': trend_name}
        if seasonal_period is not None:
            kwargs['seasonal'] = seasonal_period

        model = UnobservedComponents(**kwargs)
        res = model.fit(disp=False, maxiter=300)
        pred = pd.Series(
            res.forecast(steps=len(y_valid), exog=pd.DataFrame(X_valid).astype(float)),
            index=y_valid.index
        ).astype(float)

        aligned = safe_metric_frame(y_valid, pred)
        if aligned.empty:
            return None

        return {'trend_name': trend_name,'seasonal_period': seasonal_period,'rmse': float(np.sqrt(mean_squared_error(aligned['y_true'], aligned['y_pred']))),'aic': getattr(res, 'aic', np.nan),
            'bic': getattr(res, 'bic', np.nan),'result': res,'forecast': pred}
    except Exception:
        return None


def tune_ucm(endog_train, exog_train):
    y_fit = endog_train.iloc[:-VALIDATION_SIZE]
    y_val = endog_train.iloc[-VALIDATION_SIZE:]
    X_fit = exog_train.iloc[:-VALIDATION_SIZE]
    X_val = exog_train.iloc[-VALIDATION_SIZE:]

    trend_candidates = ['local level', 'local linear trend']
    seasonal_candidates = [None, 12, 24]

    records = []
    candidates = []

    for trend_name, seasonal_period in product(trend_candidates, seasonal_candidates):
        fitted = fit_ucm_candidate(y_fit, X_fit, y_val, X_val, trend_name, seasonal_period)
        if fitted is None:
            continue

        records.append({'trend_name': trend_name,'seasonal_period': seasonal_period,'rmse': fitted['rmse'],'aic': fitted['aic'],'bic': fitted['bic']})
        candidates.append(fitted)

    tuning_df = pd.DataFrame(records).sort_values(['rmse', 'bic', 'aic']).reset_index(drop=True)
    best_candidate = min(candidates, key=lambda item: (item['rmse'], item['bic'], item['aic']))
    return best_candidate, tuning_df


def fit_tvp_arx_rls(y_train, X_train, y_valid=None, X_valid=None, lam=0.99, p_y=1, x_lag=0, ridge_scale=25.0):
    y_train = pd.Series(y_train).astype(float).copy()
    X_train = pd.DataFrame(X_train).astype(float).copy()
    feature_cols = list(X_train.columns)

    max_lag = max(p_y, x_lag)

    y_mean = float(y_train.mean())
    y_std = float(y_train.std(ddof=0))
    if y_std == 0 or np.isnan(y_std):
        y_std = 1.0

    x_mean = X_train.mean()
    x_std = X_train.std(ddof=0).replace(0, 1.0).fillna(1.0)

    y_scaled = (y_train - y_mean) / y_std
    X_scaled = (X_train - x_mean) / x_std

    design_rows = []
    targets = []
    design_index = []

    for t in range(max_lag, len(y_train)):
        row = [1.0]
        for lag in range(1, p_y + 1):
            row.append(float(y_scaled.iloc[t - lag]))

        for feature in feature_cols:
            x_pos = t - x_lag
            row.append(float(X_scaled[feature].iloc[x_pos]))

        design_rows.append(row)
        targets.append(float(y_scaled.iloc[t]))
        design_index.append(y_train.index[t])

    Z = np.asarray(design_rows, dtype=float)
    target_arr = np.asarray(targets, dtype=float)

    k = Z.shape[1]
    beta_path = np.zeros((len(target_arr), k))
    P = np.eye(k) * ridge_scale

    for t in range(len(target_arr)):
        x_t = Z[t]
        y_t = target_arr[t]

        if t == 0:
            beta_prev = np.zeros(k)
        else:
            beta_prev = beta_path[t - 1].copy()

        R = P @ x_t
        S = lam + x_t.T @ R
        K = R / S
        error_t = y_t - x_t @ beta_prev
        beta_curr = beta_prev + K * error_t
        P = (P - np.outer(K, R)) / lam
        beta_path[t] = beta_curr

    beta_final = beta_path[-1].copy()

    fitted_scaled = pd.Series(Z @ beta_final, index=design_index)
    fitted_values = fitted_scaled * y_std + y_mean

    out = {'beta_path': beta_path,'beta_final': beta_final,'feature_cols': feature_cols,'p_y': p_y,'x_lag': x_lag,'lam': lam,'ridge_scale': ridge_scale,'y_mean': y_mean,'y_std': y_std,
        'x_mean': x_mean,'x_std': x_std,'fitted_values': fitted_values}

    if y_valid is not None and X_valid is not None:
        forecast = tvp_arx_forecast(state=out,y_history=y_train,X_history=X_train,X_future=X_valid)
        out['forecast'] = forecast
        out['rmse'] = rmse(y_valid, forecast)

    return out


def tvp_arx_forecast(state, y_history, X_history, X_future):
    y_history = pd.Series(y_history).astype(float).copy()
    X_history = pd.DataFrame(X_history).astype(float).copy()
    X_future = pd.DataFrame(X_future).astype(float).copy()

    feature_cols = state['feature_cols']
    beta_final = state['beta_final']
    p_y = state['p_y']
    x_lag = state['x_lag']
    y_mean = state['y_mean']
    y_std = state['y_std']
    x_mean = state['x_mean']
    x_std = state['x_std']

    exog_combined = pd.concat([X_history[feature_cols], X_future[feature_cols]], axis=0)
    y_buffer = list(y_history.values.astype(float))
    preds = []

    for h in range(len(X_future)):
        row = [1.0]

        for lag in range(1, p_y + 1):
            y_lag_value = y_buffer[-lag]
            row.append((y_lag_value - y_mean) / y_std)

        for feature in feature_cols:
            combined_pos = len(X_history) + h - x_lag
            x_value = float(exog_combined[feature].iloc[combined_pos])
            row.append((x_value - x_mean[feature]) / x_std[feature])

        row = np.asarray(row, dtype=float)
        pred_scaled = float(row @ beta_final)
        pred_level = pred_scaled * y_std + y_mean

        preds.append(pred_level)
        y_buffer.append(pred_level)

    return pd.Series(preds, index=X_future.index)


def tune_tvp_arx(endog_train, exog_train):
    y_fit = endog_train.iloc[:-VALIDATION_SIZE]
    y_val = endog_train.iloc[-VALIDATION_SIZE:]
    X_fit = exog_train.iloc[:-VALIDATION_SIZE]
    X_val = exog_train.iloc[-VALIDATION_SIZE:]

    top_feature_options = sorted(set([
        min(len(selected_features), 4),
        min(len(selected_features), 6),
        len(selected_features)]))
    lambda_candidates = [0.95, 0.97, 0.99]
    p_candidates = [1, 2, 3]
    x_lag_candidates = [0, 1]

    records = []
    candidates = []

    for top_k, lam, p_y, x_lag in product(top_feature_options, lambda_candidates, p_candidates, x_lag_candidates):
        feat_cols = selected_features[:top_k]
        try:
            fitted = fit_tvp_arx_rls(y_fit,X_fit[feat_cols],y_valid=y_val,X_valid=X_val[feat_cols],lam=lam,p_y=p_y,x_lag=x_lag,ridge_scale=25.0)
            records.append({'top_k_features': top_k,'lam': lam,'p_y': p_y,'x_lag': x_lag,'rmse': fitted['rmse']})
            fitted['feat_cols'] = feat_cols
            candidates.append(fitted)
        except Exception:
            continue



    tuning_df = pd.DataFrame(records).sort_values('rmse').reset_index(drop=True)
    best_candidate = min(candidates, key=lambda item: item['rmse'])
    return best_candidate, tuning_df


In [ ]:
stationarity_train_df = make_stationarity_table(train[var_cols], max_d=1)
integration_orders = stationarity_train_df.set_index('series')['chosen_diff_order'].astype(int).to_dict()

exog_diff_order_map = {col: int(integration_orders.get(col, 0)) for col in selected_features}

target_diff_order = int(integration_orders.get('target_ratio', 0))

endog_train_stationary, endog_test_stationary = difference_series_train_test(endog_train,endog_test,diff_order=target_diff_order)

exog_train_stationary, exog_test_stationary = build_stationary_train_test(exog_train,exog_test,diff_order_map=exog_diff_order_map)

endog_train_stationary, exog_train_stationary = align_target_and_exog(endog_train_stationary, exog_train_stationary)
endog_test_stationary, exog_test_stationary = align_target_and_exog(endog_test_stationary, exog_test_stationary)

arima_train_df = pd.concat([endog_train.rename('target_ratio'), exog_train_stationary], axis=1).dropna()
arima_endog_train = arima_train_df['target_ratio'].astype(float)
arima_exog_train = arima_train_df[selected_features].astype(float)
arima_exog_test = exog_test_stationary[selected_features].astype(float)

reg_train_df = pd.concat([endog_train_stationary.rename('target_ratio'), exog_train_stationary], axis=1).dropna()
reg_test_df = pd.concat([endog_test_stationary.rename('target_ratio'), exog_test_stationary], axis=1).dropna()
reg_endog_train = reg_train_df['target_ratio'].astype(float)
reg_exog_train = reg_train_df[selected_features].astype(float)
reg_endog_test = reg_test_df['target_ratio'].astype(float)
reg_exog_test = reg_test_df[selected_features].astype(float)

display(stationarity_train_df)


In [ ]:
best_arimax_spec, best_sarimax_spec, arimax_tuning_df, sarimax_tuning_df, arima_diff_diag_df = tune_arimax_sarimax(endog_train=arima_endog_train,exog_train=arima_exog_train)

arimax_model = SARIMAX(arima_endog_train,exog=arima_exog_train,order=best_arimax_spec['order'],seasonal_order=(0, 0, 0, 0),trend=best_arimax_spec['trend'],enforce_stationarity=False,enforce_invertibility=False)
arimax_res = arimax_model.fit(disp=False, maxiter=200)
arimax_forecast = pd.Series(arimax_res.forecast(steps=len(arima_exog_test), exog=arima_exog_test),index=arima_exog_test.index).astype(float)

sarimax_model = SARIMAX(arima_endog_train,exog=arima_exog_train,order=best_sarimax_spec['order'],seasonal_order=best_sarimax_spec['seasonal_order'],trend=best_sarimax_spec['trend'],
    enforce_stationarity=False,enforce_invertibility=False)
sarimax_res = sarimax_model.fit(disp=False, maxiter=200)
sarimax_forecast = pd.Series(sarimax_res.forecast(steps=len(arima_exog_test), exog=arima_exog_test),index=arima_exog_test.index).astype(float)

print(best_arimax_spec['order'])

display(arima_diff_diag_df)
display(arimax_tuning_df.head(10))
display(sarimax_tuning_df.head(10))


In [ ]:
best_var_spec, var_tuning_df, var_diag = tune_var_system(var_train, selected_features)
var_system_cols = ['target_ratio'] + selected_features
var_stationarity_df = var_diag['stationarity_table'].copy()
best_var_diff_order = int(best_var_spec['diff_order'])
best_var_lag_order = int(best_var_spec['lag_order'])

if best_var_diff_order == 0:
    var_model_data = var_train[var_system_cols].copy()
    var_model = VAR(var_model_data)
    var_res = var_model.fit(best_var_lag_order)
    var_pred = var_res.forecast(var_model_data.values[-var_res.k_ar:], steps=len(test))
    var_forecast = pd.Series(var_pred[:, 0], index=test.index).astype(float)
else:
    var_model_data = var_train[var_system_cols].diff(best_var_diff_order).dropna()
    var_model = VAR(var_model_data)
    var_res = var_model.fit(best_var_lag_order)
    var_pred_diff = var_res.forecast(var_model_data.values[-var_res.k_ar:], steps=len(test))
    var_forecast = pd.Series(forecast_from_differences(var_train['target_ratio'].iloc[-1], var_pred_diff[:, 0]),index=test.index).astype(float)

var_is_stable = bool(var_res.is_stable())
var_root_modulus = np.abs(var_res.roots)

print(best_var_lag_order)
print(best_var_diff_order)

display(var_stationarity_df)
display(var_tuning_df.head(12))


In [ ]:

best_vecm_spec, vecm_tuning_df = tune_vecm_system(var_train, selected_features)

vecm_model = VECM(var_train[['target_ratio'] + selected_features],k_ar_diff=best_vecm_spec['k_ar_diff'],coint_rank=best_vecm_spec['coint_rank'],deterministic='co')
vecm_res = vecm_model.fit()
vecm_pred = vecm_res.predict(steps=len(test))
vecm_forecast = pd.Series(vecm_pred[:, 0], index=test.index).astype(float)

print( best_vecm_spec['k_ar_diff'])
print( best_vecm_spec['coint_rank'])
display(vecm_tuning_df.head(10))


ПРи ошибке в данной ячейке можно просто запустить следующую без пезепасука ядра

In [ ]:

best_ardl_spec, ardl_tuning_df = tune_ardl(endog_train, exog_train)

ardl_feat_cols = best_ardl_spec['feat_cols']
ardl_model = ARDL(endog_train,lags=best_ardl_spec['p_lag'],exog=exog_train[ardl_feat_cols],order={feature: best_ardl_spec['q_lag'] for feature in ardl_feat_cols},
    trend='c',causal=best_ardl_spec['causal'])
ardl_res = ardl_model.fit()
ardl_forecast = pd.Series(ardl_res.forecast(steps=len(test), exog=exog_test[ardl_feat_cols]),index=test.index).astype(float)

print('p lag', best_ardl_spec['p_lag'])
print('q lag', best_ardl_spec['q_lag'])

display(ardl_tuning_df.head(12))

best_uecm_spec, uecm_tuning_df = tune_uecm(endog_train, exog_train)
uecm_feat_cols = best_uecm_spec['feat_cols']
uecm_ardl_model = ARDL(endog_train,lags=best_uecm_spec['p_lag'],exog=exog_train[uecm_feat_cols],order={feature: best_uecm_spec['q_lag'] for feature in uecm_feat_cols},
    trend='c',causal=best_uecm_spec['causal'])
uecm_model = UECM.from_ardl(uecm_ardl_model)
uecm_res = uecm_model.fit()
uecm_forecast = pd.Series(uecm_res.forecast(steps=len(test), exog=exog_test[uecm_feat_cols]),index=test.index).astype(float)

print('p lag', best_uecm_spec['p_lag'])
print('q lag', best_uecm_spec['q_lag'])

display(uecm_tuning_df.head(12))



In [ ]:
UCM_SPEC = {'level': 'local linear trend','seasonal': 24}

ucm_model = UnobservedComponents(endog_train.astype(float),level=UCM_SPEC['level'],seasonal=UCM_SPEC['seasonal'],exog=exog_train.astype(float))

ucm_res = ucm_model.fit(disp=False, maxiter=500)
ucm_forecast = pd.Series( ucm_res.forecast(steps=len(test), exog=exog_test.astype(float)),index=test.index,name='UCM').astype(float)


In [ ]:
best_tvp_spec, tvp_tuning_df = tune_tvp_arx(reg_endog_train, reg_exog_train)

feat_cols = best_tvp_spec['feat_cols']
tvp_final_state = fit_tvp_arx_rls(reg_endog_train,reg_exog_train[feat_cols],lam=best_tvp_spec['lam'],p_y=best_tvp_spec['p_y'],x_lag=best_tvp_spec['x_lag'],ridge_scale=best_tvp_spec['ridge_scale'])

beta = tvp_final_state['beta_path']
beta_final = tvp_final_state['beta_final']

tvp_forecast_stationary = tvp_arx_forecast(state=tvp_final_state,y_history=reg_endog_train,X_history=reg_exog_train[feat_cols],X_future=reg_exog_test[feat_cols]).astype(float)

if target_diff_order == 0:
    tvp_forecast = tvp_forecast_stationary.reindex(test.index)
else:
    tvp_forecast = pd.Series(forecast_from_differences(endog_train.iloc[-1], tvp_forecast_stationary.values),index=tvp_forecast_stationary.index).reindex(test.index)

print('лямбда', best_tvp_spec['lam'])
print('p_y', best_tvp_spec['p_y'])
print('x лаг', best_tvp_spec['x_lag'])

display(tvp_tuning_df.head(12))


In [ ]:
def _safe_get_arimax_desc():
    try:
        return str(best_arimax_spec)
    except:
        try:
            return str(MODEL_CONFIG.get('ARIMAX', {}))
        except:
            return "ошибся в коде с тоступом к модели arimax"

def _safe_get_var_desc():
    parts = []
    try:
        if 'diff_order' in best_var_spec:
            parts.append(f"diff_order={best_var_spec['diff_order']}")
        if 'lag_order' in best_var_spec:
            parts.append(f"lag_order={best_var_spec['lag_order']}")
        elif 'lags' in best_var_spec:
            parts.append(f"lag_order={best_var_spec['lags']}")
    except:
        try:
            cfg = MODEL_CONFIG.get('VAR', {})
            if 'diff_order' in cfg:
                parts.append(f"diff_order={cfg['diff_order']}")
            if 'lags' in cfg:
                parts.append(f"lag_order={cfg['lags']}")
        except:
            pass
    return ", ".join(parts) if parts else "ошибся в коде с тоступом к модели var"

def _safe_get_vecm_desc():
    parts = []
    try:
        if 'k_ar_diff' in best_vecm_spec:
            parts.append(f"k_ar_diff={best_vecm_spec['k_ar_diff']}")
        if 'coint_rank' in best_vecm_spec:
            parts.append(f"coint_rank={best_vecm_spec['coint_rank']}")
        if 'deterministic' in best_vecm_spec:
            parts.append(f"deterministic={best_vecm_spec['deterministic']}")
    except:
        try:
            cfg = MODEL_CONFIG.get('VECM', {})
            if 'k_ar_diff' in cfg:
                parts.append(f"k_ar_diff={cfg['k_ar_diff']}")
            if 'coint_rank' in cfg:
                parts.append(f"coint_rank={cfg['coint_rank']}")
            if 'deterministic' in cfg:
                parts.append(f"deterministic={cfg['deterministic']}")
        except:
            pass
    return ", ".join(parts) if parts else "ошибся в коде с тоступом к модели vecm"

def _safe_get_ardl_desc():
    parts = []
    try:
        if 'p_lag' in best_ardl_spec:
            parts.append(f"p={best_ardl_spec['p_lag']}")
        if 'q_lag' in best_ardl_spec:
            parts.append(f"q={best_ardl_spec['q_lag']}")
        if 'causal' in best_ardl_spec:
            parts.append(f"causal={best_ardl_spec['causal']}")
    except:
        try:
            cfg = MODEL_CONFIG.get('ARDL', {})
            if 'p_lag' in cfg:
                parts.append(f"p={cfg['p_lag']}")
            if 'q_lag' in cfg:
                parts.append(f"q={cfg['q_lag']}")
            if 'causal' in cfg:
                parts.append(f"causal={cfg['causal']}")
        except:
            pass
    return ", ".join(parts) if parts else "ошибся в коде с тоступом к модели ardl"

def _safe_get_uecm_desc():
    parts = []
    try:
        if 'p_lag' in best_uecm_spec:
            parts.append(f"p={best_uecm_spec['p_lag']}")
        if 'q_lag' in best_uecm_spec:
            parts.append(f"q={best_uecm_spec['q_lag']}")
        if 'causal' in best_uecm_spec:
            parts.append(f"causal={best_uecm_spec['causal']}")
    except:
        try:
            cfg = MODEL_CONFIG.get('UECM', {})
            if 'p_lag' in cfg:
                parts.append(f"p={cfg['p_lag']}")
            if 'q_lag' in cfg:
                parts.append(f"q={cfg['q_lag']}")
            if 'causal' in cfg:
                parts.append(f"causal={cfg['causal']}")
        except:
            pass
    return ", ".join(parts) if parts else "ошибся в коде с доступом к модели ucm"

def _safe_get_ucm_desc():
    return "локальный тренд 24 сезонность"

def _safe_get_tvp_desc():
    parts = []
    try:
        if 'lam' in best_tvp_spec:
            parts.append(f"lambda={best_tvp_spec['lam']}")
        elif 'lambda' in best_tvp_spec:
            parts.append(f"lambda={best_tvp_spec['lambda']}")
        if 'p_y' in best_tvp_spec:
            parts.append(f"y_lags={best_tvp_spec['p_y']}")
        elif 'y_lags' in best_tvp_spec:
            parts.append(f"y_lags={best_tvp_spec['y_lags']}")
        if 'x_lag' in best_tvp_spec:
            parts.append(f"x_lag={best_tvp_spec['x_lag']}")
        if 'max_features' in best_tvp_spec:
            parts.append(f"max_features={best_tvp_spec['max_features']}")
    except:
        try:
            cfg = MODEL_CONFIG.get('TVP-ARX', {})
            if 'lambda' in cfg:
                parts.append(f"lambda={cfg['lambda']}")
            if 'y_lags' in cfg:
                parts.append(f"y_lags={cfg['y_lags']}")
            if 'x_lag' in cfg:
                parts.append(f"x_lag={cfg['x_lag']}")
            if 'max_features' in cfg:
                parts.append(f"max_features={cfg['max_features']}")
        except:
            pass
    return ", ".join(parts) if parts else "ошибся в коде с тоступом к модели tvp arx"

selected_hyperparams_df = pd.DataFrame([
    {'Model': 'ARIMAX','выбранные параметры': _safe_get_arimax_desc()},
    {'Model': 'VAR','выбранные параметры': _safe_get_var_desc()},
    {'Model': 'VECM','выбранные параметры': _safe_get_vecm_desc()},
    {'Model': 'ARDL','выбранные параметры': _safe_get_ardl_desc()},
    {'Model': 'ARDL-ECM','выбранные параметры': _safe_get_uecm_desc()},
    {'Model': 'UCM','выбранные параметры': _safe_get_ucm_desc()},
    {'Model': 'TVP-ARX','выбранные параметры': _safe_get_tvp_desc()}])

display(selected_hyperparams_df)

In [ ]:

model_preds = {'ARIMAX': arimax_forecast,'SARIMAX': sarimax_forecast,'VAR': var_forecast,
    'VECM': vecm_forecast,'ARDL': ardl_forecast,'UCM': ucm_forecast,'TVP-ARX': tvp_forecast}

metrics = []
for name, pred in model_preds.items():
    aligned = safe_metric_frame(endog_test, pred)
    if aligned.empty:
        metrics.append({'Model': name, 'MAE': np.nan, 'RMSE': np.nan, 'n_eval': 0})
        continue
    mae = mean_absolute_error(aligned['y_true'], aligned['y_pred'])
    rmse_value = np.sqrt(mean_squared_error(aligned['y_true'], aligned['y_pred']))
    metrics.append({'Model': name, 'MAE': mae, 'RMSE': rmse_value, 'n_eval': len(aligned)})

metrics_df = pd.DataFrame(metrics).sort_values('RMSE', na_position='last').reset_index(drop=True)
display(metrics_df)


In [ ]:

plot_metrics = metrics_df.set_index('Model')[['MAE', 'RMSE']]
fig, axes = plt.subplots(4, 2, figsize=(15, 18), sharex=True)
axes = axes.flatten()
for ax, (name, pred) in zip(axes, model_preds.items()):
    ax.plot(train.index, endog_train, label='реальный трейн', color='blue')
    ax.plot(test.index, endog_test, label='реальный тест', color='black', linewidth=2)
    ax.plot(test.index, pred, label=f'{name} forecast', color='red')
    ax.axvline(test.index[0], color='gray', linestyle='--', linewidth=1)
    ax.set_title(name)
    if name in plot_metrics.index:
        mae_val = plot_metrics.loc[name, 'MAE']
        rmse_val = plot_metrics.loc[name, 'RMSE']
        label_text = f"MAE = {mae_val:.6f}\nRMSE = {rmse_val:.6f}"
        ax.text(0.02, 0.98,label_text,transform=ax.transAxes,ha='left', va='top',bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3, linestyle='--')

for ax in axes[len(model_preds):]:
    ax.axis('off')

fig.suptitle('сравнение прогнозов моделей', fontsize=16, y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.985])

figure_path = 'final_model_сравнение.png'
fig.savefig(figure_path, dpi=300, bbox_inches='tight')
plt.show()


display(metrics_df)

In [ ]:
OUTPUT_DIR = Path("article_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
from statsmodels.tsa.stattools import adfuller

def make_adf_table(series_dict):
    records = []
    for name, s in series_dict.items():
        clean = s.dropna()
        if len(clean) < 10:
            records.append((name, None, None, None, None))
            continue
        res = adfuller(clean, autolag='AIC')
        records.append((name, res[0], res[1], res[2], res[3]))
    return pd.DataFrame(records, columns=['Series', 'ADF_stat', 'p-value', 'lags', 'nobs'])

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred)
    denom = np.where(denom == 0, 1e-12, denom)
    return 100 * np.mean(2 * np.abs(y_pred - y_true) / denom)

def mape_safe(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return 100 * np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps)))

def directional_accuracy(y_true, y_pred):
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)

    dy_true = y_true.diff().dropna()
    dy_pred = y_pred.diff().dropna()

    common_idx = dy_true.index.intersection(dy_pred.index)
    if len(common_idx) == 0:
        return np.nan

    s_true = np.sign(dy_true.loc[common_idx])
    s_pred = np.sign(dy_pred.loc[common_idx])
    return 100 * np.mean(s_true == s_pred)





def to_aligned_series(pred, target_index, model_name):


    if isinstance(pred, pd.DataFrame):

        pred = pred.iloc[:, 0]

    if isinstance(pred, pd.Series):
        pred_values = pred.astype(float).to_numpy().reshape(-1)
    else:
        pred_values = np.asarray(pred, dtype=float).reshape(-1)

    min_len = min(len(pred_values), len(target_index))


    return pd.Series(pred_values[:min_len], index=target_index[:min_len], name=model_name).astype(float)

def safe_metric_frame(y_true, y_pred):
    tmp = pd.concat([pd.Series(y_true, name='y_true').astype(float),pd.Series(y_pred, name='y_pred').astype(float)], axis=1).dropna()
    return tmp
candidate_models = [("ARIMAX", ["arimax_forecast", "arima_forecast"]),("VAR", ["var_forecast"]),
    ("VECM", ["vecm_forecast"]),("ARDL", ["ardl_forecast"]),
    ("ARDL-ECM", ["uecm_forecast"]),("UCM", ["ucm_forecast"]),("TVP-ARX", ["tvp_forecast"]),]

model_preds = {}
missing_models = []

for pretty_name, var_candidates in candidate_models:
    found = False
    for var_name in var_candidates:
        if var_name in globals():
            try:
                s = to_aligned_series(globals()[var_name], test.index, pretty_name)
                model_preds[pretty_name] = s
                found = True
                break
            except Exception as e:
                print(f"ошибка с глобалами")
    if not found:
        missing_models.append(pretty_name)




metrics_rows = []
aligned_store = {}

for model_name, pred in model_preds.items():
    aligned = safe_metric_frame(endog_test, pred)

    if aligned.empty:
        metrics_rows.append({"Model": model_name,"N_used": 0,"MAE": np.nan,"RMSE": np.nan,"sMAPE_%": np.nan,"MAPE_safe_%": np.nan,"Directional_Accuracy_%": np.nan})
        continue

    aligned_store[model_name] = aligned.copy()

    metrics_rows.append({"Model": model_name,"N_used": len(aligned),"MAE": mean_absolute_error(aligned["y_true"], aligned["y_pred"]),"RMSE": rmse(aligned["y_true"], aligned["y_pred"]),
        "sMAPE_%": smape(aligned["y_true"], aligned["y_pred"]),"MAPE_safe_%": mape_safe(aligned["y_true"], aligned["y_pred"]),"Directional_Accuracy_%": directional_accuracy(aligned["y_true"], aligned["y_pred"])})

metrics_df = pd.DataFrame(metrics_rows).sort_values("RMSE", na_position="last").reset_index(drop=True)
metrics_df["ранжировка_RMSE"] = np.arange(1, len(metrics_df) + 1)
metrics_df = metrics_df[["ранжировка_RMSE", "Model", "N_used", "MAE", "RMSE", "sMAPE_%", "MAPE_safe_%", "Directional_Accuracy_%"]]

display(metrics_df)
forecast_table = pd.DataFrame(index=test.index)
forecast_table["Actual"] = endog_test.astype(float)

for model_name, pred in model_preds.items():
    aligned_pred = to_aligned_series(pred, test.index, model_name)
    forecast_table[model_name] = aligned_pred.reindex(test.index)

display(forecast_table.head())
error_table = pd.DataFrame(index=test.index)
error_table["Actual"] = endog_test.astype(float)

for model_name in model_preds.keys():
    pred_aligned_full = forecast_table[model_name]
    error_table[f"{model_name}Err"] = error_table["Actual"] - pred_aligned_full
    error_table[f"{model_name}AE"] = np.abs(error_table["Actual"] - pred_aligned_full)

display(error_table.head())

sample_info = pd.DataFrame({"Block": ["Full sample", "Train", "Test"],"Start": [series.index.min(), train.index.min(), test.index.min()],
    "End": [series.index.max(), train.index.max(), test.index.max()],"Observations": [len(series), len(train), len(test)]})

desc_vars = ["target_ratio", "rate_spread"]
desc_vars = [v for v in desc_vars if v in series.columns]

desc_full = series[desc_vars].describe().T
desc_train = train[desc_vars].describe().T
desc_test = test[desc_vars].describe().T

display(sample_info)
display(desc_full)
'''
Код ниже был написан нейросетью open.ai chat GPT
Также названия были скорректированы мною , то есть код сгенерированный с моими правками
Код был сгенерирован по промту : исходя из кода что я тебе прислал, сделай мне красивые графики и сохрани их для того , чтобы я мог использовать их  в своей ВКР, а также сохрани итоговые резуьлтаты моделирования в отдельные удобные файлы

Модель: deepseek,expert,deepthink https://chat.deepseek.com

Работа выполнена правильно на 7 из 10, не все графики поулчились очень красивыми и удобными для вставки в вкр
'''
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import pandas as pd
def save_latex_table(df, path, caption=None, label=None, float_format="%.6f"):
    fmt = float_format.__mod__ if isinstance(float_format, str) else None
    latex = df.to_latex(index=True, escape=False, float_format=fmt)

    if caption or label:
        wrapper = []
        wrapper.append("\\begin{table}[!htbp]\n\\centering\n")
        if caption:
            wrapper.append(f"\\caption{{{caption}}}\n")
        if label:
            wrapper.append(f"\\label{{{label}}}\n")
        wrapper.append(latex)
        wrapper.append("\n\\end{table}\n")
        latex = "".join(wrapper)

    with open(path, "w", encoding="utf-8") as f:
        f.write(latex)
stationarity_dict = {"target_ratio_level": series["target_ratio"],"target_ratio_diff1": series["target_ratio"].diff(),"target_ratio_diff2": series["target_ratio"].diff().diff(),}

if "rate_spread" in series.columns:
    stationarity_dict["rate_spread_level"] = series["rate_spread"]

stationarity_df = make_adf_table(stationarity_dict)
display(stationarity_df)
overlay_path = OUTPUT_DIR / "figure_1.png"

plt.figure(figsize=(13, 7))
plt.plot(train.index, endog_train, label="реальный трейн")
plt.plot(test.index, endog_test, label="реальный тест", linewidth=2)

for model_name, pred in forecast_table.drop(columns=["Actual"]).items():
    plt.plot(pred.index, pred.values, label=model_name)

plt.axvline(test.index[0], linestyle="--")
plt.title("сравнение прогнозов")
plt.xlabel("Дата")
plt.ylabel("Таргет")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(overlay_path, dpi=300, bbox_inches="tight")
plt.show()
subplots_path = OUTPUT_DIR / "figure_2"

n_models = len(model_preds)
ncols = 2
nrows = int(np.ceil(n_models / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(14, 4.5 * nrows))
axes = np.array(axes).reshape(-1)

for ax, model_name in zip(axes, model_preds.keys()):
    pred = forecast_table[model_name]
    ax.plot(train.index, endog_train, label="реальный трейн")
    ax.plot(test.index, endog_test, label="реальный тест", linewidth=2)
    ax.plot(pred.index, pred.values, label=model_name)
    ax.axvline(test.index[0], linestyle="--")
    ax.set_title(model_name)
    ax.legend(fontsize=7)

for j in range(len(model_preds), len(axes)):
    axes[j].axis("off")

fig.tight_layout()
fig.savefig(subplots_path, dpi=300, bbox_inches="tight")
plt.show()
metrics_bar_path = OUTPUT_DIR / "figure_3"

plot_df = metrics_df.sort_values("RMSE", na_position="last")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(plot_df["Model"], plot_df["RMSE"])
axes[0].set_title("RMSE по моделям")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(plot_df["Model"], plot_df["MAE"])
axes[1].set_title("MAE по моделям")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
fig.savefig(metrics_bar_path, dpi=300, bbox_inches="tight")
plt.show()
abs_error_path = OUTPUT_DIR / "figure_4"

plt.figure(figsize=(13, 7))
for model_name in model_preds.keys():
    plt.plot(error_table.index, error_table[f"{model_name}AE"], label=model_name)

plt.title("абсолютная ошибка на тесте")
plt.xlabel("Дата")
plt.ylabel("ошибка")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(abs_error_path, dpi=300, bbox_inches="tight")
plt.show()
valid_metrics = metrics_df.dropna(subset=["RMSE"])


best_model_name = valid_metrics.iloc[0]["Model"]
best_pred = forecast_table[best_model_name]

best_model_path = OUTPUT_DIR / "figure_5"

plt.figure(figsize=(12, 6))
plt.plot(train.index, endog_train, label="тренировочная реальная")
plt.plot(test.index, endog_test, label="тренировочная реальная", linewidth=2)
plt.plot(best_pred.index, best_pred.values, label=f"Forecast: {best_model_name}", linewidth=2)
plt.axvline(test.index[0], linestyle="--")
plt.title(f"лучшая модель по RMSE {best_model_name}")
plt.xlabel("Дата")
plt.ylabel("Таргет")
plt.legend()
plt.tight_layout()
plt.savefig(best_model_path, dpi=300, bbox_inches="tight")
plt.show()
summary_candidates = {
    "arimax_summary.txt": ["arimax_res", "arima_res"],
    "var_summary.txt": ["var_res"],
    "vecm_summary.txt": ["vecm_res"],
    "ardl_summary.txt": ["ardl_res"],
    "ucm_summary.txt": ["ucm_res"],
}

for filename, obj_names in summary_candidates.items():
    for obj_name in obj_names:
        if obj_name in globals():
            try:
                txt = str(globals()[obj_name].summary())
                with open(OUTPUT_DIR / filename, "w", encoding="utf-8") as f:
                    f.write(txt)
                break
            except Exception:
                pass

forecast_table.to_csv(OUTPUT_DIR / "forecast_table.csv", index=True)
error_table.to_csv(OUTPUT_DIR / "error_table.csv", index=True)
metrics_df.to_csv(OUTPUT_DIR / "metrics_table.csv", index=False)
sample_info.to_csv(OUTPUT_DIR / "sample_info.csv", index=False)
stationarity_df.to_csv(OUTPUT_DIR / "stationarity_tests.csv", index=False)

with pd.ExcelWriter(OUTPUT_DIR / "article_tables.xlsx") as writer:
    metrics_df.to_excel(writer, sheet_name="metrics", index=False)
    forecast_table.to_excel(writer, sheet_name="forecasts", index=True)
    error_table.to_excel(writer, sheet_name="errors", index=True)
    sample_info.to_excel(writer, sheet_name="sample_info", index=False)
    stationarity_df.to_excel(writer, sheet_name="stationarity", index=False)
    desc_full.to_excel(writer, sheet_name="desc_full", index=True)

save_latex_table(metrics_df.set_index("RMSE"), OUTPUT_DIR / "metrics_table.tex",
                 caption="точность прогнозов", label="metrics")
save_latex_table(sample_info.set_index("Block"), OUTPUT_DIR / "sample_info.tex",
                 caption="sample", label="sample")
save_latex_table(stationarity_df.set_index("Series"), OUTPUT_DIR / "stationarity_тест.tex",
                 caption="стационарность", label="стационарность")
